# Frequentist Inference Case Study - Part B

## Learning objectives

Welcome to Part B of the Frequentist inference case study! The purpose of this case study is to help you apply the concepts associated with Frequentist inference in Python. In particular, you'll practice writing Python code to apply the following statistical concepts: 
* the _z_-statistic
* the _t_-statistic
* the difference and relationship between the two
* the Central Limit Theorem, including its assumptions and consequences
* how to estimate the population mean and standard deviation from a sample
* the concept of a sampling distribution of a test statistic, particularly for the mean
* how to combine these concepts to calculate a confidence interval

In the previous notebook, we used only data from a known normal distribution. **You'll now tackle real data, rather than simulated data, and answer some relevant real-world business problems using the data.**

## Hospital medical charges

Imagine that a hospital has hired you as their data scientist. An administrator is working on the hospital's business operations plan and needs you to help them answer some business questions. 

In this assignment notebook, you're going to use frequentist statistical inference on a data sample to answer the questions:
* has the hospital's revenue stream fallen below a key threshold?
* are patients with insurance really charged different amounts than those without?

Answering that last question with a frequentist approach makes some assumptions, and requires some knowledge, about the two groups.

We are going to use some data on medical charges obtained from [Kaggle](https://www.kaggle.com/easonlai/sample-insurance-claim-prediction-dataset). 

For the purposes of this exercise, assume the observations are the result of random sampling from our single hospital. Recall that in the previous assignment, we introduced the Central Limit Theorem (CLT), and its consequence that the distributions of sample statistics approach a normal distribution as $n$ increases. The amazing thing about this is that it applies to the sampling distributions of statistics that have been calculated from even highly non-normal distributions of data! Recall, also, that hypothesis testing is very much based on making inferences about such sample statistics. You're going to rely heavily on the CLT to apply frequentist (parametric) tests to answer the questions in this notebook.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import t
from numpy.random import seed
medical = pd.read_csv('data/insurance2.csv')

In [ ]:
medical.shape

In [ ]:
medical.head()

__Q1:__ Plot the histogram of charges and calculate the mean and standard deviation. Comment on the appropriateness of these statistics for the data.

__A:__ The mean of charges is about 13,270usd with a standard deviation of about 12,110usd. But the histogram shows charges are heavily right-skewed — most patients are billed under 15,000usd while a long tail of expensive cases stretches past 60,000usd. For a distribution this skewed, the mean and standard deviation aren't a great summary; the median and IQR would describe a typical patient better, and the standard deviation overstates the variability of a typical bill because of the long-tail influence. That said, thanks to the CLT, the mean is still a useful statistic when we shift to asking questions about the *average* charge across many patients — we just shouldn't use mean ± std as a description of where individual bills fall.

In [ ]:
_ = plt.hist(medical['charges'], bins=30)
_ = plt.xlabel('charges ($)')
_ = plt.ylabel('number of patients')
_ = plt.title('Distribution of medical charges')
_ = plt.axvline(medical['charges'].mean(), color='r', linestyle='--', label='mean')
_ = plt.axvline(medical['charges'].median(), color='g', linestyle='--', label='median')
_ = plt.legend()

In [ ]:
mean_charge = medical['charges'].mean()
std_charge = medical['charges'].std()  # pandas uses ddof=1 by default
print(f'Mean charge:   ${mean_charge:,.2f}')
print(f'Std (ddof=1):  ${std_charge:,.2f}')
print(f'Median:        ${medical["charges"].median():,.2f}')

__Q2:__ The administrator is concerned that the actual average charge has fallen below 12,000, threatening the hospital's operational model. On the assumption that these data represent a random sample of charges, how would you justify that these data allow you to answer that question? And what would be the most appropriate frequentist test, of the ones discussed so far, to apply?

__A:__ The data we have is one sample of 1,338 patient charges, which the prompt tells us to treat as a random sample. Even though the distribution itself is far from normal, the Central Limit Theorem tells us that the *sampling distribution of the mean* is approximately normal for a sample this large, so we can construct a confidence interval (or run a hypothesis test) around the mean using the standard frequentist machinery. We don't know the true population standard deviation — we estimated it from the sample — so the appropriate test is the **one-sample t-test** rather than a z-test. (With n=1338 the t-distribution is essentially indistinguishable from the normal anyway, but using t is the technically correct choice when σ is unknown.)

__Q3:__ Given the nature of the administrator's concern, what is the appropriate confidence interval in this case? A ***one-sided*** or ***two-sided*** interval? (Refresh your understanding of this concept on p. 399 of the *AoS*). Calculate the critical value and the relevant 95% confidence interval for the mean, and comment on whether the administrator should be concerned.

__A:__ The administrator only cares whether the mean has fallen *below* \$12,000 — they wouldn't be worried if it had gone up. That's a one-sided concern, so we want a one-sided 95% confidence interval: a lower bound below which we have only 5% probability that the true mean sits. Concretely, that's the 95th percentile of the t-distribution (one-tailed), giving a critical value of about 1.646 with df=1337.

The lower 95% confidence bound on the mean comes out to about 12,725usd — well *above* the 12,000usd threshold. The administrator has no statistical basis for concern: even at the pessimistic end of our interval, the true average charge appears to be safely above \$12,000.

In [ ]:
from scipy.stats import t

n = len(medical)
df = n - 1

# One-sided 95% critical value: we want the 95th percentile of the t-distribution
t_crit = t.ppf(0.95, df)
print(f'One-sided critical t value (df={df}): {t_crit:.4f}')

In [ ]:
# Standard error of the mean and one-sided margin of error
std_err = std_charge / np.sqrt(n)
margin = t_crit * std_err
print(f'Standard error: {std_err:.4f}')
print(f'Margin (one-sided): {margin:.4f}')

In [ ]:
# One-sided lower confidence bound
lower_bound = mean_charge - margin
print(f'Sample mean:                  ${mean_charge:,.2f}')
print(f'95% one-sided lower bound:    ${lower_bound:,.2f}')
print(f'Threshold of concern:         $12,000.00')
print(f'Should the administrator worry? {"Yes" if lower_bound < 12000 else "No"}')

The administrator then wants to know whether people with insurance really are charged a different amount to those without.

__Q4:__ State the null and alternative hypothesis here. Use the _t_-test for the difference between means, where the pooled standard deviation of the two groups is given by:
\begin{equation}
s_p = \sqrt{\frac{(n_0 - 1)s^2_0 + (n_1 - 1)s^2_1}{n_0 + n_1 - 2}}
\end{equation}

and the *t*-test statistic is then given by:

\begin{equation}
t = \frac{\bar{x}_0 - \bar{x}_1}{s_p \sqrt{1/n_0 + 1/n_1}}.
\end{equation}

(If you need some reminding of the general definition of ***t-statistic***, check out the definition on p. 404 of *AoS*). 

What assumption about the variances of the two groups are we making here?

__A:__ Let $\mu_0$ be the mean charge for patients *without* insurance and $\mu_1$ for patients *with* insurance.

- **Null hypothesis ($H_0$):** $\mu_0 = \mu_1$ — the two groups are charged the same on average.
- **Alternative hypothesis ($H_1$):** $\mu_0 \ne \mu_1$ — the two groups are charged different amounts on average.

The pooled standard deviation formula above bakes in the assumption that the **two groups have equal population variances** (homoscedasticity). If that assumption is badly wrong, we'd want Welch's t-test instead (which doesn't pool the variances). For now we'll proceed under the equal-variance assumption as the question directs.

__Q5:__ Perform this hypothesis test both manually, using the above formulae, and then using the appropriate function from [scipy.stats](https://docs.scipy.org/doc/scipy/reference/stats.html#statistical-tests) (hint, you're looking for a function to perform a _t_-test on two independent samples). For the manual approach, calculate the value of the test statistic and then its probability (the p-value). Verify you get the same results from both.

__A:__ 

In [ ]:
# Split into insured (insuranceclaim==1) and uninsured (insuranceclaim==0)
insured = medical.loc[medical['insuranceclaim'] == 1, 'charges']
uninsured = medical.loc[medical['insuranceclaim'] == 0, 'charges']

n0, n1 = len(uninsured), len(insured)
x0, x1 = uninsured.mean(), insured.mean()
s0, s1 = uninsured.std(ddof=1), insured.std(ddof=1)

print(f'Uninsured: n={n0}, mean=${x0:,.2f}, std=${s0:,.2f}')
print(f'Insured:   n={n1}, mean=${x1:,.2f}, std=${s1:,.2f}')

In [ ]:
# Pooled standard deviation
sp = np.sqrt(((n0 - 1) * s0**2 + (n1 - 1) * s1**2) / (n0 + n1 - 2))
print(f'Pooled std: {sp:.4f}')

# Test statistic
t_stat = (x0 - x1) / (sp * np.sqrt(1/n0 + 1/n1))
print(f'Manual t-statistic: {t_stat:.4f}')

# Two-tailed p-value using the t-distribution with df = n0 + n1 - 2
df_pool = n0 + n1 - 2
p_value = 2 * t.sf(abs(t_stat), df_pool)
print(f'Manual p-value:    {p_value:.4e}')

In [ ]:
# Now using scipy's built-in. equal_var=True gives the pooled (Student) t-test;
# equal_var=False would give Welch's version.
from scipy import stats
scipy_result = stats.ttest_ind(uninsured, insured, equal_var=True)
print(f'scipy t-statistic: {scipy_result.statistic:.4f}')
print(f'scipy p-value:    {scipy_result.pvalue:.4e}')

In [ ]:
# Verify the manual and scipy versions match
print('Match on t-statistic?', np.isclose(t_stat, scipy_result.statistic))
print('Match on p-value?    ', np.isclose(p_value, scipy_result.pvalue))

Congratulations! Hopefully you got the exact same numerical results. This shows that you correctly calculated the numbers by hand. Secondly, you used the correct function and saw that it's much easier to use. All you need to do is pass your data to it.

__Q6:__ Conceptual question: look through the documentation for statistical test functions in scipy.stats. You'll see the above _t_-test for a sample, but can you see an equivalent one for performing a *z*-test from a sample? Comment on your answer.

__A:__ There's no one-line `scipy.stats.ztest_ind` analogue to `ttest_ind`. The scipy.stats module gives you `norm.cdf` / `norm.sf` to compute z-based probabilities by hand, but no packaged "z-test from a sample" function. The reason is practical: a z-test from a sample requires the *population* standard deviation, which you almost never actually have. In almost every real situation you're estimating σ from your data, which is exactly the case the t-test was designed for. When n is large the t-distribution converges to the normal anyway, so the t-test gracefully covers both regimes. If a true z-test is needed (e.g. proportions), it lives over in `statsmodels` as `statsmodels.stats.weightstats.ztest`.

## Learning outcomes

Having completed this project notebook, you now have good hands-on experience:
* using the central limit theorem to help you apply frequentist techniques to answer questions that pertain to very non-normally distributed data from the real world
* performing inference using such data to answer business questions
* forming a hypothesis and framing the null and alternative hypotheses
* testing this using a _t_-test